In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import ast
import streamlit as st
import plotly.express as px

#Configurações do Streamlit
st.set_page_config(page_title="Dashboard de Vendas da Steam", layout="wide")
st.title("Análise de Jogos Mais Vendidos na Steam")

# Carregamento dos dados
@st.cache_data
def load_data():
    df = pd.read_csv("dataset_final.csv")
    
    # Converter estimated_downloads para numérico
    df['estimated_downloads'] = pd.to_numeric(df['estimated_downloads'], errors='coerce').fillna(0)
    
    # Criar uma cópia para métricas de jogos
    df_games = df.drop_duplicates(subset=['game_name']).copy()
    
    # Expandir para análise de tags, sistemas operacionais, linguas e recursos
    columns_to_explode = ['user_defined_tags', 'supported_languages', 'supported_os', 'other_features']
    df_exploded = df.copy()
    for col in columns_to_explode:
        if col in df_exploded.columns:
            df_exploded[col] = df_exploded[col].astype(str).str.split(',')
            df_exploded = df_exploded.explode(col)
            df_exploded[col] = df_exploded[col].str.strip()
            
    return df_exploded, df_games

# Carregar os dois dataframes
df_exploded, df_games = load_data()

#Filtros na barra lateral
st.sidebar.header("Filtros de tamanho dos gráficos")
top_tags_filtradas = st.sidebar.slider("Top Tags Definidas por Usuários", 5, 20, 10)
top_devs_filtradas = st.sidebar.slider("Top Desenvolvedores", 5, 20, 10)
top_vendas_filtradas = st.sidebar.slider("Top Jogos por Vendas", 5, 20, 10)

# 1. Top Tags, Desenvolvedores e Vendas
st.header("Top Tags, Desenvolvedores e Vendas")
col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("Top Jogos por Vendas")
    # vendas em ordem decrescente
    top_vendas = df_games.groupby('game_name')['estimated_downloads'].sum().sort_values(ascending=False).head(top_vendas_filtradas)
    st.bar_chart(top_vendas, sort="-estimated_downloads")

with col2:
    st.subheader("Top Desenvolvedores")
    # desenvolvedores em ordem decrescente
    top_devs = df_games['developer'].value_counts().head(top_devs_filtradas)
    st.bar_chart(top_devs, sort="-count")

with col3:
     st.subheader("Top Tags Definidas por Usuários")
    #tags em ordem decrescente
     top_tags = df_exploded['user_defined_tags'].value_counts().head(top_tags_filtradas)
     st.bar_chart(top_tags, sort="-count")
   
# 2. Relacoes
st.header("Análise de Relações")
col4, col5 = st.columns(2)

with col4:
    # Relação entre Dificuldade e duração da gameplay 
    sns.set_style("darkgrid")
    st.subheader("Dificuldade vs Duração da Gameplay")
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(data=df_games,x='difficulty',y='length',ax=ax,color='#1f77b4', errorbar=None)
    ax.set_title('Análise: Dificuldade vs Duração da Gameplay', fontsize=16, fontweight='bold', pad=20)
    st.pyplot(fig)

with col5:
    # Relação entre Preço e Vendas
    sns.set_style("darkgrid")
    st.subheader("Preço vs Vendas")
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.scatterplot(data=df_games,x='price',y='estimated_downloads',ax=ax,color='#1f77b4',s=100,alpha=0.7)
    ax.set_title('Análise: Preço vs Vendas Estimadas', fontsize=16, fontweight='bold', pad=20)
    st.pyplot(fig)

#Vendas por Rating
st.subheader("Vendas por Rating")
rating_sales = df_games.groupby('rating')['estimated_downloads'].sum().sort_values(ascending=False).head(10)
st.bar_chart(rating_sales, sort="-estimated_downloads")

#Analise de Distribuição
st.header("Análise de Distribuição")

# Distribuição dos sistemas operacionais
top_os = df_exploded['supported_os'].value_counts().head(3)
fig_os = px.pie(values=top_os.values, names=top_os.index, title="Distribuição dos Sistemas Operacionais Suportados")
st.plotly_chart(fig_os, use_container_width=True)

# Distribuição dos recursos suportados
top_features = df_exploded['other_features'].value_counts().head(10)
fig_features = px.pie(values=top_features.values, names=top_features.index, title="Distribuição dos Top 10 Recursos Suportados")
st.plotly_chart(fig_features, use_container_width=True)

# Exibir os dados brutos
if st.checkbox("Mostrar dados brutos"):
    st.dataframe(df_games)